In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

import tensorflow.keras as keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import zipfile
import os
import numpy as np
from PIL import Image

In [ ]:
# Extract zip files
with zipfile.ZipFile("/dataset/grayscale-train.zip", "r") as zip_ref:
    zip_ref.extractall("/dataset/train")
with zipfile.ZipFile("/dataset/grayscale-test.zip", "r") as zip_ref:
    zip_ref.extractall("/dataset/test")

# Load data
def load_data(base_path):
    classes = sorted([d for d in os.listdir(base_path)
                      if os.path.isdir(os.path.join(base_path, d))])
    images, labels = [], []

    for label, cls in enumerate(classes):
        folder = os.path.join(base_path, cls)
        for img_name in os.listdir(folder):
            img_path = os.path.join(folder, img_name)
            img = Image.open(img_path)
            img_array = np.array(img)
            images.append(img_array)
            labels.append(label)

    return np.array(images), np.array(labels)

# Load train and test sets
train_images, train_labels = load_data("dataset/train/grayscale-train/train")
test_images, test_labels = load_data("dataset/test/grayscale-test/test")

In [ ]:
# Display images
def show_samples(images, labels, samples_per_class=5):
    classes = np.unique(labels)
    num_classes = len(classes)
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(samples_per_class * 2, num_classes * 2))

    for row, cls in enumerate(classes):
        idxs = np.where(labels == cls)[0][:samples_per_class]
        for col, idx in enumerate(idxs):
            ax = axes[row, col] if num_classes > 1 else axes[col]
            ax.imshow(images[idx], cmap="gray")
            ax.set_title(f"Class {cls}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

print("Train set")
show_samples(train_images, train_labels, samples_per_class=5)

print("Test set")
show_samples(test_images, test_labels, samples_per_class=5)

In [ ]:
num_classes = 10

# input image dimensions
img_rows, img_cols = 28, 28

x_train = train_images
x_test = test_images

# Convert class vectors to binary class matrices
y_train = train_labels
y_test = test_labels

if K.image_data_format() == 'channels_first':
    x_train = x_train.reshape(x_train.shape[0], 1, img_rows, img_cols)
    x_test = x_test.reshape(x_test.shape[0], 1, img_rows, img_cols)
    input_shape = (1, img_rows, img_cols)
else:
    x_train = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
    x_test = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1)
    input_shape = (img_rows, img_cols, 1)

# Scale pixels
x_train =  x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# create the data generator to feed the training with augmentations
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train)

# Print number of samples
train_samples = x_train.shape[0]
test_samples = x_test.shape[0]
print("Train samples:", train_samples)
print("Test samples:", test_samples)

In [ ]:
models = []
def cnnFresh(cnn_epochs = 5, cnn_batch_size = 64):
    steps_per_epoch = train_samples // cnn_batch_size

    # Data augmentation
    cnn_model = Sequential()
    cnn_model.add(Conv2D(32, kernel_size=(3, 3),
                     activation='relu',
                     input_shape=input_shape))
    cnn_model.add(BatchNormalization())
    cnn_model.add(Conv2D(64, (3, 3), activation='relu'))
    cnn_model.add(BatchNormalization())
    cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
    cnn_model.add(Dropout(0.2))

    cnn_model.add(Conv2D(256, (3,3), activation='relu'))
    cnn_model.add(BatchNormalization())
    cnn_model.add(MaxPooling2D(pool_size=(2,2)))
    cnn_model.add(Dropout(0.15))

    cnn_model.add(Flatten())
    cnn_model.add(Dense(128, activation='relu'))
    cnn_model.add(BatchNormalization())
    cnn_model.add(Dropout(0.3))
    cnn_model.add(Dense(num_classes, activation='softmax'))

    cnn_model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    history_cnn = cnn_model.fit(datagen.flow(x_train, y_train,shuffle=True),
                                validation_data=(x_test, y_test),
                                epochs=cnn_epochs,steps_per_epoch=steps_per_epoch)
    models.append(cnn_model)
    return history_cnn

In [ ]:
# Run models and evaluate accuracy
histories = []
histories.append(cnnFresh(20, 64))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 0 Test Accuracy:", test_acc)

histories.append(cnnFresh(20, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 1 Test Accuracy:", test_acc)

histories.append(cnnFresh(30, 64))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 2 Test Accuracy:", test_acc)

histories.append(cnnFresh(30, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 3 Test Accuracy:", test_acc)

histories.append(cnnFresh(40, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 4 Test Accuracy:", test_acc)

histories.append(cnnFresh(50, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 5 Test Accuracy:", test_acc)

histories.append(cnnFresh(70, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 6 Test Accuracy:", test_acc)

histories.append(cnnFresh(100, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 7 Test Accuracy:", test_acc)

histories.append(cnnFresh(200, 32))
loss, test_acc = models[-1].evaluate(x_test, y_test)
print("Model 8 Test Accuracy:", test_acc)

In [ ]:
for i, history in enumerate(histories):
    # Get accuracy
    model = models[i]
    y_pred = np.argmax(model.predict(x_test), axis=1)
    test_acc = accuracy_score(y_test, y_pred)
    print(f"Model {i} Test Accuracy:", test_acc)

    cm = confusion_matrix(y_test, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Accuracy plot
    acc = history.history["accuracy"]
    axes[0].plot(acc)
    axes[0].set_title(f"Model {i} Training Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].grid(True)

    # Confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=axes[1], colorbar=False)
    axes[1].set_title(f"Confusion Matrix")

    plt.tight_layout()
    plt.show()